# 03 — Galileo (SaaS)

Same shared agent (`shared/workflow.py`) — same three scenarios — traced through **Galileo**.

**Instrumentation approach**: Galileo ships a LangChain callback handler (`galileo.handlers.langchain.GalileoCallback`) plus a `@log` decorator for non-LangChain code. We pass the callback into the graph's `config["callbacks"]`, the same way we did for Langfuse.

**Hosting**: SaaS only. Free tier: no credit card, starter trace+evaluation quota per month. Sign up at <https://app.galileo.ai/sign-up>.

What to look for in the UI after running this:
1. <https://app.galileo.ai> → project named by `GALILEO_PROJECT` in your `.env` (defaults to `observability-comparison`) → log stream named by `GALILEO_LOG_STREAM` (defaults to `default`)
2. Three top-level traces, one per scenario
3. Click a trace → the tree view shows the agent / tool spans
4. Galileo's distinguishing feature: the **Insights / Metrics** tab — built-in eval metrics like *Tool Selection Quality*, *Action Completion*, *Context Adherence* run automatically over each trace (this is the platform's bet: evaluation as a first-class citizen, not an afterthought)

## 1. Load environment

If you don't have a Galileo key yet, follow the signup steps in `README.md` → **Galileo** section. The free tier is enough for this demo.

In [ ]:
import os, sys, pathlib
from dotenv import load_dotenv

ROOT = pathlib.Path().resolve().parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

load_dotenv(ROOT / ".env")

assert os.environ.get("ANTHROPIC_API_KEY"), "Set ANTHROPIC_API_KEY in .env"
assert os.environ.get("GALILEO_API_KEY"), "Set GALILEO_API_KEY in .env (see README signup steps)"

os.environ.setdefault("GALILEO_PROJECT", "observability-comparison")
os.environ.setdefault("GALILEO_LOG_STREAM", "default")
os.environ.setdefault("GALILEO_CONSOLE_URL", "https://app.galileo.ai")

print("Galileo project    :", os.environ["GALILEO_PROJECT"])
print("Galileo log stream :", os.environ["GALILEO_LOG_STREAM"])
print("Galileo console    :", os.environ["GALILEO_CONSOLE_URL"])

## 2. Wire up the Galileo callback handler

Like Langfuse, Galileo provides a LangChain `CallbackHandler`. It reads the API key / project / log stream from the environment.

In [ ]:
from galileo.handlers.langchain import GalileoCallback

galileo_handler = GalileoCallback()
print("Galileo callback ready")

## 3. Build the shared agent — **prompt pulled from Galileo, not code**

The agent's system prompt was pushed to Galileo's project-scoped templates by `notebooks/00_setup_prompts.ipynb`. This cell fetches it back from **Galileo** at runtime — the Python `SYSTEM_PROMPT` constant in `shared/workflow.py` is not used here. Galileo stores prompts as typed `Message` objects scoped to a project (unlike Langfuse / LangSmith, which are workspace-global).

The printed `fetched N chars` line below is the proof: that content came over the wire from `GALILEO_CONSOLE_URL`, not from disk.

In [ ]:
from shared.workflow import build_agent, SCENARIOS, fetch_system_prompt

# Pull the system prompt from Galileo's project-scoped templates.
# The prompt was pushed by notebooks/00_setup_prompts.ipynb — run that
# once before this notebook (or check Templates in the Galileo UI).
print("Pulling system prompt from Galileo Templates...")
prompt_text = fetch_system_prompt("galileo")
print(f"  fetched {len(prompt_text)} chars; first line: {prompt_text.splitlines()[0]!r}")

agent = build_agent(prompt_source="galileo")
for sc in SCENARIOS:
    print(f"  {sc['id']:<12} expected_tool_calls={sc['expected_tool_calls']}  prompt={sc['prompt']!r}")

## 4. Run the three scenarios

Same pattern as the other two notebooks: invoke the graph with the handler in `config["callbacks"]`. Galileo will create one trace per `agent.invoke()` call.

In [ ]:
from langchain_core.messages import HumanMessage

# Group all 3 scenarios under one Galileo session so they appear together.
SESSION_ID = "permission-checks-demo"

results = []
for sc in SCENARIOS:
    config = {
        "callbacks": [galileo_handler],
        "run_name": sc["id"],
        "tags": ["observability_comparison", "galileo", sc["id"]],
        "metadata": {
            "scenario_id": sc["id"],
            "expected_tool_calls": sc["expected_tool_calls"],
            # Session grouping (Galileo reads `session_id`)
            "session_id": SESSION_ID,
            "thread_id": SESSION_ID,
            "langfuse_session_id": SESSION_ID,
        },
    }
    out = agent.invoke({"messages": [HumanMessage(content=sc["prompt"])]}, config=config)
    actual = sum(len(getattr(m, "tool_calls", []) or []) for m in out["messages"])
    final = out["messages"][-1].content
    results.append({**sc, "final": final, "actual_tool_calls": actual})
    print(f"\n--- {sc['id']} ---")
    print(f"prompt   : {sc['prompt']}")
    print(f"tools    : expected={sc['expected_tool_calls']}  actual={actual}")
    print(f"final    : {final}")

# Galileo's callback buffers spans; this flushes before the cell ends.
galileo_handler.flush() if hasattr(galileo_handler, "flush") else None
print(f"\nDone. Traces are in session '{SESSION_ID}'. Inspect at", os.environ["GALILEO_CONSOLE_URL"])

## 5. Galileo's headline feature: built-in agent metrics

Unlike Langfuse and LangSmith, Galileo runs an out-of-the-box evaluation suite **on every trace, automatically**. Open one of the traces and look at the **Insights** panel — you'll see scores for:

| Metric | What it measures |
| --- | --- |
| **Tool Selection Quality** | Did the agent pick the right tools for the user's request? |
| **Action Completion** | Did the multi-step task finish successfully? |
| **Context Adherence** | Did the model stay grounded in the tool outputs vs. hallucinate? |
| **Instruction Adherence** | Did it follow the system prompt? |

These run server-side after ingestion — you don't write any eval code. That's the bet: the value isn't the trace tree (everyone has one), it's the automatic scoring of agent quality.

If you want a numeric programmatic score on top (like we did for Langfuse / LangSmith), Galileo exposes that too:

In [ ]:
# Galileo's programmatic logging API. The exact attribute / method names
# can shift between SDK versions, so we keep this in a try/except so the
# notebook stays runnable even if the API surface drifts.
try:
    from galileo import log

    @log(span_type="workflow", name="scoring_sweep")
    def scored_run(prompt: str, expected: int) -> dict:
        out = agent.invoke(
            {"messages": [HumanMessage(content=prompt)]},
            config={"callbacks": [GalileoCallback()]},
        )
        actual = sum(len(getattr(m, "tool_calls", []) or []) for m in out["messages"])
        return {
            "actual_tool_calls": actual,
            "expected_tool_calls": expected,
            "tool_call_count_matches": int(actual == expected),
        }

    for sc in SCENARIOS:
        print(sc["id"], scored_run(sc["prompt"], sc["expected_tool_calls"]))
except ImportError:
    print("`from galileo import log` not available in this SDK version — skipping programmatic scoring.")

## Takeaways for the comparison

- **Setup cost**: one `GalileoCallback()`, drop into `config["callbacks"]`. Same shape as Langfuse.
- **What you get out of the box**: trace tree + **automatic agent-quality metrics** (Tool Selection Quality, Action Completion, etc.) on every trace — no eval code required.
- **Where it leans in**: evaluation-first; built-in agent metrics, hallucination / groundedness scoring, guardrails for production.
- **Friction**: SaaS only; free tier has trace/eval quotas; newer SDK so API surface changes more often than LangSmith's.